# 96 - Gradio avanzado: servir un modelo de imágenes — Base

Interfaz **Gradio** para un modelo de clasificación de imágenes en Keras (demo MNIST).

Esta celda comprueba si están disponibles las dependencias necesarias para la demo de imágenes: Gradio, NumPy y TensorFlow/Keras.


In [ ]:
HAS_DEPS=True
try:
    import gradio as gr, numpy as np
    from tensorflow import keras
    from tensorflow.keras import layers
    print('✅ Dependencias importadas')
except Exception as e:
    HAS_DEPS=False; print('⚠️ Faltan dependencias:', e)


Esta celda crea una arquitectura CNN mínima para MNIST. Si TensorFlow no está disponible o falla la creación del modelo, se deja `model=None` para usar una salida segura.


In [ ]:
model=None
if HAS_DEPS:
    try:
        model=keras.Sequential([
            layers.Input(shape=(28,28,1)),
            layers.Conv2D(8,(3,3),activation='relu'),
            layers.MaxPooling2D(),
            layers.Flatten(),
            layers.Dense(10,activation='softmax')
        ])
        model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
        print('Modelo Keras creado (sin entrenar).')
    except Exception as e:
        print('No se pudo crear el modelo:', e)


Esta celda define la función de predicción que recibirá la imagen desde Gradio, la preprocesará y devolverá probabilidades por dígito.


In [ ]:
def predict(img):
    import numpy as np
    if model is None:
        return {str(i):0.0 for i in range(10)}
    img=img.convert('L').resize((28,28))
    x=np.array(img,dtype='float32')/255.0
    x=x.reshape(1,28,28,1)
    probs=model.predict(x,verbose=0)[0]
    return {str(i):float(probs[i]) for i in range(10)}


Esta celda construye la interfaz Gradio que conecta la entrada de imagen con la función `predict`, pero no lanza el servidor automáticamente.


In [ ]:
if HAS_DEPS:
    demo=gr.Interface(
        fn=predict,
        inputs=gr.Image(type='pil', shape=(28,28), image_mode='L'),
        outputs=gr.Label(num_top_classes=3),
        title='Clasificador MNIST (demo)'
    )
    print('Ejecuta demo.launch() para abrir la interfaz.')


## Ampliación progresiva

Para que una demo de imágenes sea útil hay que controlar más cosas que el `predict` básico:

1. Preprocesado reproducible.
2. Salida top-k interpretable.
3. Ejemplos de prueba aunque no haya dataset externo.
4. Información de depuración para detectar errores de formato.

Esta celda separa el preprocesado de la imagen y añade una predicción con información de depuración. Separar estas piezas facilita detectar errores de forma, escala o tipo de dato.


In [ ]:
def preprocess_mnist_image(img):
    import numpy as np
    img = img.convert('L').resize((28, 28))
    x = np.array(img, dtype='float32') / 255.0
    return x.reshape(1, 28, 28, 1)

def predict_with_debug(img):
    x = preprocess_mnist_image(img)
    if model is None:
        probs = np.zeros(10)
    else:
        probs = model.predict(x, verbose=0)[0]
    top3 = sorted({str(i): float(probs[i]) for i in range(10)}.items(), key=lambda item: item[1], reverse=True)[:3]
    debug = {"shape": tuple(x.shape), "min": float(x.min()), "max": float(x.max())}
    return dict(top3), debug

Esta celda crea una imagen sintética de ejemplo y, si es posible, monta una interfaz `Blocks` con salida top-k y JSON de depuración.


In [ ]:
if HAS_DEPS:
    try:
        from PIL import Image, ImageDraw
        sample = Image.new('L', (28, 28), color=0)
        draw = ImageDraw.Draw(sample)
        draw.line((8, 4, 18, 14, 10, 24), fill=255, width=3)
        print("Imagen sintética creada para probar predict_with_debug:", predict_with_debug(sample))
    except Exception as e:
        print("No se pudo crear imagen sintética:", e)

### Reto adicional

Crea una interfaz con dos salidas: etiqueta top-k y JSON de depuración. Añade ejemplos sintéticos o imágenes del dataset que entrenaste en clase.

## Para profundizar

Este notebook muestra Gradio para imágenes. La **documentación completa** (`Gradio_Documentacion.md`) incluye más funcionalidades avanzadas:

- **Tipos de input**: Image, Audio, Video, File, Dataframe, etc.
- **Eventos**: change, click, submit, upload.
- **Blocks API**: interfaz personalizada con filas, columnas, pestañas.
- **Estado**: gr.State() para mantener valores entre llamadas.
- **Streaming**: генерация token a token para texto.
- **Despliegue**: Hugging Face Spaces, Docker, API REST.
- **Ejemplos avanzados**: múltiples inputs, validación, progress bars.

Para ver más ejemplos de Blocks y funcionalidades avanzadas, consulta `Gradio_Documentacion.md`.